In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

#Calcualtes the genetic vector size based on the one-hot econding array
#risk_score = total number of unique genes
genetic_input_dim = 1 + len(unique_genes)

class MultimodalGNN(nn.Module):
    """Late fusion GNN for the multi-modal toxicity regression
    Combines atomic node messaging with one-hot encoded genetic data vectors
    to scale the baseline toxicity score
    """
    def __init__(self, num_node_features=1):
        """Initializes layers for graph convolutions and dense linear fusion
        num_node_features - size of the graph node attributes 
        """
        #Initializes the parent nn.Module calss 
        super(MultimodalGNN, self).__init__()

        #Compresses raw node features into a 16-dimensional (feature) layer
        self.conv1 = GCNConv(num_node_features, 16)
        #Expands into 32 features  
        self.conv2 = GCNConv(16, 32)
        #Outputs the final 64 features per atom
        self.conv3 = GCNConv(32, 64)

        #Accepts the 64 features + the genetic vector size
        self.fc1 = nn.Linear(64 + genetic_input_dim, 32)
        #Shrinks the hidden data from 32 down to 16 features
        self.fc2 = nn.Linear(32, 16)
        #Turns the 16 feautres into a single toxicity score 
        self.out = nn.Linear(16, 1)

    def forward(self, data):
        """
        Implements the forward pass by convolving nodes, pooling graph, 
        fusing the data, and scaling the output
        data - PyTorch Geometric data object containing x, edge_index, 
        batch, and genetic_attr
        Returns a tensor containing a scaled rating prediction bounded 
        between 0.0 and 100.0
        """
        # Unpacks the graph components (atoms, bonds, graph IDs)
        x, edge_index, batch = data.x, data.edge_index, data.batch
        genetic_attr = data.genetic_attr

        # Passes the atoms and bonds through the 3 convolution layers 
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))

        # Averages the features of all individual atoms inside the model
        molecule_vector = global_mean_pool(x, batch)

        # Combines the 64 drug numbers and the risk number into one row
        fused_vector = torch.cat([molecule_vector, genetic_attr], dim=1)

        # Passes the combined drug-patient data through final linear layers 
        x = F.relu(self.fc1(fused_vector))
        x = F.relu(self.fc2(x))
        
        #Scaling final output
        raw_output = self.out(x)
        return torch.sigmoid(raw_output) * 100.0

model = MultimodalGNN()